In [1]:
import torch
import numpy as np
import random
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CheckpointCallback
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

fragment_pool = pd.read_csv("source/SMILES_list.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]
kmax = 5
totalsteps = 200000

def make_env():
    return OPSDesignerEnv(
        smiles=fragment_pool,
        target_ops=targetOPS,
        min_k=5,
        max_k=kmax,
        max_steps=kmax+1,
        eps=1e-8,
        r_cap=100,
    )

vec_env = make_vec_env(make_env, n_envs=1, seed=42)

model = PPO("MlpPolicy",
            vec_env,
            seed=42,
            verbose=1,
            device="cpu",
            batch_size=50,
            n_steps=1000,
           )

checkpoint_callback = CheckpointCallback(save_freq=10000, save_path="./RLmodels/", name_prefix="ppo_2566")
model.learn(total_timesteps=totalsteps, callback=checkpoint_callback)
model.save(f"RLmodels/ppo_2566_{totalsteps}_final.zip")

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 5        |
|    ep_rew_mean     | 0.523    |
| time/              |          |
|    fps             | 3        |
|    iterations      | 1        |
|    time_elapsed    | 285      |
|    total_timesteps | 1000     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 5           |
|    ep_rew_mean          | 0.0183      |
| time/                   |             |
|    fps                  | 3           |
|    iterations           | 2           |
|    time_elapsed         | 565         |
|    total_timesteps      | 2000        |
| train/                  |             |
|    approx_kl            | 0.035079148 |
|    clip_fraction        | 0.347       |
|    clip_range           | 0.2         |
|    entropy_loss         | -7.85       |
|    explained_variance   | -2.71e-05   |
|    learning